# Genetic algorithm optimization

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

A genetic algorithm takes an existing larva model, varies some of its parameters within defined
bounds, and keeps whatever generation after generation gets closer to a reference dataset. What
"closer" means is exactly the error measured in the previous lesson.

Said in one sentence it sounds simple; in practice four decisions have to be made, and this
notebook is organized around them :

1. **Which model** is being optimized, and **which of its parameters** may vary - the *optimization
   space*.
2. **How each generation is built** from the previous one - population size, elitism, mutation.
3. **What counts as good** - the reference dataset and the evaluation metrics.
4. **How long** it runs - number of generations and duration of each simulation.

The last section runs one, but only if you switch it on : even a short GA is many simulations.

**What you will be able to do afterwards**

- Build an optimization space from whole modules, or narrow it to named parameters.
- Set the selection parameters and say what each one costs.
- Reuse one of the stored GA configurations and override just what you need.
- Have the best genome stored automatically as a new model configuration.

**Prerequisites** : [Model evaluation](model_evaluation.ipynb).

**Cost** : nothing until you set `RUN_GA_DEMO = True`. The demo below is deliberately tiny -
one generation of ten agents for 0.1 simulated minutes.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_GA_DEMO` | `False` | the genetic algorithm itself |
| `RUN_GA_VIDEO_DEMO` | `False` | a second run that writes a video of the generations |

## Setup

In [1]:
%matplotlib inline

%load_ext param.ipython

import larvaworld as lw
from larvaworld.lib import reg
from larvaworld.lib.model.modules.module_modes import SpaceDict
from larvaworld.lib.sim.genetic_algorithm import (
    GAconf,
    GAevaluation,
    GAlauncher,
    GAselector,
)

lw.VERBOSE = 1

# Tutorial safety switches
RUN_GA_DEMO = False  # runs the genetic algorithm
RUN_GA_VIDEO_DEMO = False  # runs it again, writing a video

DEMO_EXP = "realism"
DEMO_DURATION_MIN = 0.1
DEMO_NGENERATIONS = 1
DEMO_NAGENTS = 10  # keep >= 7 when selection_ratio = 0.3
SAVE_MEDIA = False
MEDIA_DIR = "./media"

Welcome to the param IPython extension! (https://param.holoviz.org/)
Available magics: %params


Initializing larvaworld registry


Registry configured!


## Section 1 : The configuration as a whole

`GAconf` is the full GA configuration. Beside the general simulation options and the environment,
it has the two blocks that matter here : `ga_select_kws` (the selection side) and `ga_eval_kws`
(the evaluation side).

The package ships with ready-made GA experiments, callable by name through the `experiment`
argument. They are a good place to start because each one already points at a sensible reference
dataset and metric set.

In [2]:
%params GAconf

In [3]:
print("Stored GA experiments :")
for gaID in reg.conf.Ga.confIDs:
    print(f"  {gaID}")

Stored GA experiments :
  chemorbit
  exploration
  interference
  obstacle_avoidance
  realism


## Section 2 : The optimization space

The GA optimizes an existing model, named by `base_model`. Which of its parameters may vary is
decided by `space_mkeys` : you name **modules**, and every free parameter of those modules becomes
a dimension of the search space, with the bounds declared on the parameter itself.

That is the coarse way. Two further options narrow it :

- **`include_effector_params`** - by default the low-level `Effector` parameters such as
  `input_noise` and `output_noise` are excluded. Set it to `True` to reach them.
- **`space_pkeys`** - an explicit allowlist of bare parameter names, restricting the space to
  exactly those, within the modules you listed.

`init_mode` decides where the first generation comes from : `random` samples the space,
`model` starts every agent from the base model's own values, `default` from the parameter defaults.

`SpaceDict` is the class that does this, and it is worth building one by hand once - the resulting
`space_ks` is the list of things the GA will actually be varying.

In [4]:
%params SpaceDict

In [5]:
space_dict = SpaceDict(base_model="explorer", space_mkeys=["interference", "crawler"])

print(f"Optimizing {space_dict.base_model!r} over modules {space_dict.space_mkeys}")
print(f"{len(space_dict.space_ks)} free parameters :")
for k in space_dict.space_ks:
    print(f"  {k}")

Optimizing 'explorer' over modules ['interference', 'crawler']
10 free parameters :
  brain.interference.attenuation
  brain.interference.attenuation_max
  brain.interference.suppression_mode
  brain.interference.max_attenuation_phase
  brain.crawler.amp
  brain.crawler.freq
  brain.crawler.stride_dst_mean
  brain.crawler.stride_dst_std
  brain.crawler.max_vel_phase
  brain.crawler.max_scaled_vel


Narrowing the same module down to two named parameters shows the difference between the coarse and
the precise way of building a space. A smaller space converges faster and is far easier to
interpret - which is why the [worked example](ga_turner_noise_optimization.ipynb) optimizes exactly
two parameters rather than a whole module.

In [6]:
narrow = SpaceDict(
    base_model="explorer",
    space_mkeys=["turner"],
    include_effector_params=True,
    space_pkeys=["input_noise", "output_noise"],
)

print("Narrowed space :", narrow.space_ks)

Narrowed space : ['brain.turner.input_noise', 'brain.turner.output_noise']


## Section 3 : Building the next generation

`GAselector` adds the evolutionary parameters on top of the space :

| parameter | what it controls |
|---|---|
| `Ngenerations` | how many generations to run; `None` means run until stopped |
| `Nagents` | population size, i.e. how many models are simulated per generation |
| `Nelits` | how many best genomes survive unchanged into the next generation |
| `selection_ratio` | the fraction of the population that is allowed to breed |
| `Pmutation` | probability that a given parameter of an offspring mutates |
| `Cmutation` | how far a mutation may move, as a fraction of the parameter's range |
| `bestConfID` | model ID under which the best genome is stored |

`Nagents` is the one that costs : each agent is a simulation, so the total compute is roughly
`Ngenerations x Nagents x duration`. Keep `Nagents` above `1 / selection_ratio` or there will be
nothing left to breed from.

In [7]:
%params GAselector

## Section 4 : What counts as good

`GAevaluation` extends the evaluation configuration of the previous lesson with the GA-specific
part : how a genome is turned into a single fitness number.

There are two ways of scoring, and they suit different problems :

- **Against a reference dataset** - give a `refID` and `eval_metrics`, optionally
  `cycle_curve_metrics` to fit the shape of the stride cycle as well. Fitness is the distance to
  the real animals. This is what the `exploration`, `interference` and `realism` experiments do.
- **With a fitness function** - name one of the built-in `fitness_func_name` functions instead, for
  goals that are not a dataset comparison : `dst2source` rewards getting close to an odor source,
  `cum_dst` rewards covering distance. This is what `chemorbit` and `obstacle_avoidance` do.

`exclusion_mode` is the third possibility : instead of scoring, simply exclude agents that violate
a criterion, and let the survivors breed.

In [8]:
%params GAevaluation

## Section 5 : Running one

`GAlauncher` takes either a stored experiment name with overrides, or a complete `parameters`
dictionary. The first form is shown here.

The result is a dictionary holding the optimization space, the fitness achieved and the best
genome, whose model configuration is available as `mConf`.

In [9]:
ga1 = GAlauncher(
    experiment=DEMO_EXP,
    duration=DEMO_DURATION_MIN,
    screen_kws={"show_display": False},
)

ga1.selector.Ngenerations = DEMO_NGENERATIONS
ga1.selector.Nagents = DEMO_NAGENTS

print(f"Experiment      : {DEMO_EXP!r}")
print(f"Base model      : {ga1.selector.base_model!r}")
print(f"Space           : {ga1.selector.space_ks}")
print(f"Generations     : {ga1.selector.Ngenerations} x {ga1.selector.Nagents} agents")
print(f"Simulations     : {ga1.selector.Ngenerations * ga1.selector.Nagents}")

Loaded existing conf 30controls


Loaded stored reference dataset : exploration.30controls
Experiment      : 'realism'
Base model      : 'explorer'
Space           : ['brain.interference.attenuation', 'brain.interference.attenuation_max', 'brain.interference.suppression_mode', 'brain.interference.max_attenuation_phase', 'brain.turner.base_activation', 'brain.turner.activation_range', 'brain.turner.tau', 'brain.turner.w_ee', 'brain.turner.w_ce', 'brain.turner.w_ec', 'brain.turner.w_cc', 'brain.turner.m', 'brain.turner.n']
Generations     : 1 x 10 agents
Simulations     : 10


In [10]:
if RUN_GA_DEMO:
    best1 = ga1.simulate()
    print("Result keys :", best1.keylist)
    print()
    print("Best model configuration :")
    best1.mConf.print()
else:
    print("Set RUN_GA_DEMO = True to run the genetic algorithm.")

Set RUN_GA_DEMO = True to run the genetic algorithm.


### Starting from a stored configuration instead

The other form retrieves the stored configuration, modifies it, and passes it as `parameters`. It
is the more flexible route, because everything in the configuration is reachable - not only what
the launcher exposes as an argument.

The same run is set up here with video output, which renders the whole population of each
generation. It is the most direct way to see selection happening, and also the slowest.

In [11]:
p = reg.conf.Ga.expand(DEMO_EXP)

p.ga_select_kws.Ngenerations = DEMO_NGENERATIONS
p.ga_select_kws.Nagents = DEMO_NAGENTS

print("Selection settings :", p.ga_select_kws.keylist)

Selection settings : ['Cmutation', 'Nagents', 'Nelits', 'Ngenerations', 'Pmutation', 'base_model', 'bestConfID', 'init_mode', 'selection_ratio', 'space_mkeys']


In [12]:
if RUN_GA_VIDEO_DEMO:
    screen_kws = {
        "vis_mode": "video",
        "show_display": False,
        "save_video": SAVE_MEDIA,
        "media_dir": MEDIA_DIR,
        "video_file": f"ga_{DEMO_EXP}",
    }
    ga2 = GAlauncher(parameters=p, duration=DEMO_DURATION_MIN, screen_kws=screen_kws)
    best2 = ga2.simulate()
    best2.mConf.print()
else:
    print("Set RUN_GA_VIDEO_DEMO = True to run the GA with video output.")

Set RUN_GA_VIDEO_DEMO = True to run the GA with video output.


## Where to go next

- [Worked example: turner noise](ga_turner_noise_optimization.ipynb) - the whole thing end to end,
  on two parameters, with before-and-after videos and a model diff.
- [Model evaluation](model_evaluation.ipynb) - the error the GA is minimizing.
- Reference : [Genetic algorithm optimization (advanced)](../../working_with_larvaworld/ga_optimization_advanced.md)
  and [Batch runs (advanced)](../../working_with_larvaworld/batch_runs_advanced.md), the other way
  of searching a parameter space.